In [ ]:
!pip install mlflow dagshub -q

In [ ]:
!pip install pytorch-forecasting pytorch-lightning --quiet

import os
import json
import time
import random
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader

import mlflow
import dagshub

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

logging.getLogger("mlflow.pytorch").setLevel(logging.ERROR)
logging.getLogger("mlflow.utils.requirements_utils").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pl.seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

dagshub.init(repo_owner='ejoba22', repo_name='walmart-sales-forecasting', mlflow=True)
mlflow.set_experiment("TFT_Training")

print("\nTracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("TFT_Training"))

BASE = '/kaggle/input/datasets/elenejobava/walmart-features-engineered/'

train_fe = pd.read_parquet(BASE + 'train_features.parquet')
test_fe  = pd.read_parquet(BASE + 'test_features.parquet')

with open(BASE + 'feature_cols.json') as f:
    feature_cols = json.load(f)

train_fe["Date"] = pd.to_datetime(train_fe["Date"])
test_fe["Date"]  = pd.to_datetime(test_fe["Date"])

print("\nTrain:", train_fe.shape, train_fe["Date"].min(), "->", train_fe["Date"].max())
print("Test :", test_fe.shape,  test_fe["Date"].min(),  "->", test_fe["Date"].max())

TRAIN_WEEKS = 143
TEST_HORIZON = 39
CV_FOLD_HORIZON = 13
N_CV_FOLDS = 3

fold_cutoffs = []
for i in range(N_CV_FOLDS):
    val_end = TRAIN_WEEKS - (N_CV_FOLDS - 1 - i) * CV_FOLD_HORIZON
    val_start = val_end - CV_FOLD_HORIZON
    fold_cutoffs.append((val_start, val_end))

print("\nFold cutoffs (reused across all architectures):")
for i, (vs, ve) in enumerate(fold_cutoffs):
    print(f"  Fold {i}: train < week {vs}, validate on weeks [{vs}, {ve})")

train_pairs_all = set(zip(train_fe.Store, train_fe.Dept))
test_pairs_all  = set(zip(test_fe.Store, test_fe.Dept))
cold_start_pairs = list(test_pairs_all - train_pairs_all)
active_pairs = sorted(train_pairs_all)

print(f"\nActive Store-Dept series: {len(active_pairs)}")
print(f"Cold-start pairs: {len(cold_start_pairs)}")

## 1. Panel Construction with Time Index

In [ ]:
full_date_range = pd.date_range(train_fe["Date"].min(), test_fe["Date"].max(), freq="7D")

calendar_cols = [
    'IsHoliday', 'Year', 'Month', 'Week', 'DayOfYear', 'Quarter', 'Year_norm',
    'Week_sin', 'Week_cos', 'Month_sin', 'Month_cos',
    'weeks_to_SuperBowl', 'near_SuperBowl', 'before_SuperBowl',
    'weeks_to_LaborDay', 'near_LaborDay', 'before_LaborDay',
    'weeks_to_Thanksgiving', 'near_Thanksgiving', 'before_Thanksgiving',
    'weeks_to_Christmas', 'near_Christmas', 'before_Christmas',
    'is_super_bowl', 'is_labor_day', 'is_thanksgiving', 'is_christmas'
]
static_cols = ['Type_enc', 'Size_log']

store_covariates = pd.concat([
    train_fe[['Store', 'Date'] + calendar_cols + static_cols],
    test_fe[['Store', 'Date'] + calendar_cols + static_cols]
], axis=0).drop_duplicates(subset=['Store', 'Date']).sort_values(['Store', 'Date'])

target_rows = []
for store_i, dept_i in active_pairs:
    obs = train_fe[(train_fe.Store == store_i) & (train_fe.Dept == dept_i)][['Date', 'Weekly_Sales']]
    obs = obs.set_index('Date').reindex(full_date_range)
    obs['Store'] = store_i
    obs['Dept'] = dept_i
    obs['is_active'] = obs['Weekly_Sales'].notna().astype(int)
    obs['Weekly_Sales'] = obs['Weekly_Sales'].fillna(0).clip(lower=0)
    obs = obs.reset_index().rename(columns={'index': 'Date'})
    target_rows.append(obs)

target_panel = pd.concat(target_rows, axis=0, ignore_index=True)
full_panel = target_panel.merge(store_covariates, on=['Store', 'Date'], how='left')

full_panel = full_panel.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
full_panel['time_idx'] = full_panel.groupby(['Store', 'Dept']).cumcount()
full_panel['series_id'] = full_panel['Store'].astype(str) + "_" + full_panel['Dept'].astype(str)
full_panel['Store'] = full_panel['Store'].astype(str)
full_panel['Dept'] = full_panel['Dept'].astype(str)

print("Full panel shape:", full_panel.shape)
print("Nulls after merge:\n", full_panel.isnull().sum()[full_panel.isnull().sum() > 0])
print("\ntime_idx range:", full_panel['time_idx'].min(), "-", full_panel['time_idx'].max())
print("Unique series_id count:", full_panel['series_id'].nunique())
print("\nSample:")
print(full_panel[['series_id', 'Store', 'Dept', 'Date', 'time_idx', 'Weekly_Sales', 'is_active']].head())

## 2. TimeSeriesDataSet Construction

In [ ]:
MAX_ENCODER_LENGTH = 52

known_reals = calendar_cols
unknown_reals = ['Weekly_Sales']

def build_fold_datasets(fold_idx, batch_size=128):
    val_start, val_end = fold_cutoffs[fold_idx]
    train_cutoff_idx = val_start

    training_data = full_panel[full_panel.time_idx < train_cutoff_idx]
    fold_data = full_panel[full_panel.time_idx < val_end]

    training_dataset = TimeSeriesDataSet(
        training_data,
        time_idx="time_idx",
        target="Weekly_Sales",
        group_ids=["series_id"],
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=CV_FOLD_HORIZON,
        static_categoricals=["Store", "Dept"],
        static_reals=["Size_log"],
        time_varying_known_categoricals=[],
        time_varying_known_reals=known_reals,
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=unknown_reals,
        target_normalizer=GroupNormalizer(groups=["series_id"], transformation="softplus"),
        weight="is_active",  # zero-filled inactive weeks no longer count toward the loss
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True,
    )

    validation_dataset = TimeSeriesDataSet.from_dataset(
        training_dataset, fold_data, predict=True, stop_randomization=True
    )

    train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

    return training_dataset, validation_dataset, train_dataloader, val_dataloader

# sanity check on fold 0, same as your original print statements
training_dataset, validation_dataset, train_dataloader, val_dataloader = build_fold_datasets(0)
print("Training dataset length:", len(training_dataset))
print("Validation dataset length:", len(validation_dataset))
print("Train batches:", len(train_dataloader), "| Val batches:", len(val_dataloader))

In [ ]:
def compute_wmae(model, val_dataloader, full_panel, active_only=True):
    preds_out = model.predict(val_dataloader, mode="prediction", return_x=True, return_index=True)
    preds = preds_out.output.cpu().numpy()
    actuals = preds_out.x["decoder_target"].cpu().numpy()
    index_df = preds_out.index.reset_index(drop=True)

    horizon = preds.shape[1]
    rows = []
    for i in range(len(index_df)):
        series_id = index_df.loc[i, "series_id"]
        start_t = index_df.loc[i, "time_idx"]
        for h in range(horizon):
            rows.append({"series_id": series_id, "time_idx": start_t + h,
                         "pred": preds[i, h], "actual": actuals[i, h]})
    pred_df = pd.DataFrame(rows)
    pred_df = pred_df.merge(
        full_panel[["series_id", "time_idx", "IsHoliday", "is_active"]],
        on=["series_id", "time_idx"], how="left"
    )

    if active_only:
        pred_df = pred_df[pred_df["is_active"] == 1]  # ignore the artificial zero-fill weeks

    weights = np.where(pred_df["IsHoliday"] == 1, 5, 1)
    wmae = np.average(np.abs(pred_df["pred"] - pred_df["actual"]), weights=weights)
    mae = np.mean(np.abs(pred_df["pred"] - pred_df["actual"]))
    return wmae, mae, pred_df


class MLflowEpochLogger(pl.callbacks.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        for k, v in trainer.callback_metrics.items():
            if isinstance(v, torch.Tensor):
                v = v.item()
            if isinstance(v, (int, float)):
                mlflow.log_metric(f"epoch_{k.replace('/', '_')}", v, step=trainer.current_epoch)

## 3. Model Definition + Training (Fold 0)

In [ ]:
import pytorch_forecasting
print(pytorch_forecasting.__version__)

from pytorch_forecasting import TemporalFusionTransformer
print(TemporalFusionTransformer.__mro__[:5])

In [ ]:
def train_tft_fold(fold_idx, hparams, run_name, max_epochs=30, patience=5):
    training_dataset, validation_dataset, train_dataloader, val_dataloader = build_fold_datasets(
        fold_idx, batch_size=hparams.get("batch_size", 128)
    )

    tft = TemporalFusionTransformer.from_dataset(
        training_dataset,
        learning_rate=hparams["learning_rate"],
        hidden_size=hparams["hidden_size"],
        attention_head_size=hparams["attention_head_size"],
        dropout=hparams["dropout"],
        hidden_continuous_size=hparams["hidden_continuous_size"],
        loss=QuantileLoss(),
        optimizer="adam",
    )
    n_params = sum(p.numel() for p in tft.parameters())

    early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=patience, mode="min")
    checkpoint_callback = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        gradient_clip_val=hparams.get("gradient_clip_val", 0.1),
        callbacks=[early_stop_callback, checkpoint_callback, MLflowEpochLogger()],
        enable_progress_bar=True,
        logger=False,
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({"model_type": "TFT", "fold": str(fold_idx), "run_type": "cv"})
        mlflow.log_params({
            "fold": fold_idx, "max_encoder_length": MAX_ENCODER_LENGTH,
            "max_prediction_length": CV_FOLD_HORIZON, **hparams,
            "n_params": n_params, "max_epochs": max_epochs,
            "early_stopping_patience": patience,
        })

        start_time = time.time()
        trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
        elapsed = time.time() - start_time

        best_val_loss = checkpoint_callback.best_model_score.item()
        best_model_path = checkpoint_callback.best_model_path

        wmae, mae, pred_df = compute_wmae(tft, val_dataloader, full_panel)

        mlflow.log_metric("best_val_quantile_loss", best_val_loss)
        mlflow.log_metric("val_wmae", wmae)
        mlflow.log_metric("val_mae", mae)
        mlflow.log_metric("training_time_minutes", elapsed / 60)
        mlflow.log_metric("n_epochs_trained", trainer.current_epoch + 1)
        mlflow.log_artifact(best_model_path, artifact_path="checkpoints")

        print(f"[Fold {fold_idx}] {run_name}: {elapsed/60:.1f} min, "
              f"{trainer.current_epoch+1} epochs, quantile_loss={best_val_loss:.2f}, "
              f"WMAE={wmae:.2f}, MAE={mae:.2f}")

        return {"fold": fold_idx, "run_name": run_name, "val_quantile_loss": best_val_loss,
                "val_wmae": wmae, "val_mae": mae, "training_time_min": elapsed / 60,
                "n_epochs": trainer.current_epoch + 1, "checkpoint_path": best_model_path}

In [ ]:
baseline_hparams = {
    "learning_rate": 0.03, "hidden_size": 16, "attention_head_size": 1,
    "dropout": 0.1, "hidden_continuous_size": 8, "batch_size": 128,
    "gradient_clip_val": 0.1,
}

fold_results = []
for fold_idx in range(N_CV_FOLDS):
    result = train_tft_fold(fold_idx, baseline_hparams, run_name=f"TFT_Fold{fold_idx}_Baseline")
    fold_results.append(result)

results_df = pd.DataFrame(fold_results)
print(results_df[["fold", "val_quantile_loss", "val_wmae", "val_mae", "training_time_min"]])

with mlflow.start_run(run_name="TFT_Baseline_CV_Summary"):
    mlflow.set_tags({"model_type": "TFT", "run_type": "cv_summary"})
    mlflow.log_metric("cv_mean_wmae", results_df["val_wmae"].mean())
    mlflow.log_metric("cv_std_wmae", results_df["val_wmae"].std())
    mlflow.log_metric("cv_mean_quantile_loss", results_df["val_quantile_loss"].mean())
    results_df.to_csv("tft_baseline_cv_results.csv", index=False)
    mlflow.log_artifact("tft_baseline_cv_results.csv")

## 5. Epoch Loss Curves (from already-logged MLflow metrics)

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = mlflow.get_experiment_by_name("TFT_Training")
print("Experiment ID:", experiment.experiment_id)

In [ ]:
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'TFT_Fold2_Baseline'",
    order_by=["start_time DESC"], max_results=1
)

fold2_run = runs[0]
print("Run ID:", fold2_run.info.run_id)
print("Logged metrics:", fold2_run.data.metrics)

artifacts = client.list_artifacts(fold2_run.info.run_id, path="checkpoints")
print("\nArtifacts found:", [a.path for a in artifacts])

if artifacts:
    local_checkpoint_path = client.download_artifacts(fold2_run.info.run_id, artifacts[0].path, "/kaggle/working/")
    print("\nDownloaded checkpoint to:", local_checkpoint_path)

In [ ]:
BASE = '/kaggle/input/datasets/elenejobava/walmart-features-engineered/'
train_fe = pd.read_parquet(BASE + 'train_features.parquet')
test_fe  = pd.read_parquet(BASE + 'test_features.parquet')
train_fe["Date"] = pd.to_datetime(train_fe["Date"])
test_fe["Date"]  = pd.to_datetime(test_fe["Date"])

TRAIN_WEEKS = 143
CV_FOLD_HORIZON = 13
N_CV_FOLDS = 3
fold_cutoffs = []
for i in range(N_CV_FOLDS):
    val_end = TRAIN_WEEKS - (N_CV_FOLDS - 1 - i) * CV_FOLD_HORIZON
    val_start = val_end - CV_FOLD_HORIZON
    fold_cutoffs.append((val_start, val_end))

train_pairs_all = set(zip(train_fe.Store, train_fe.Dept))
active_pairs = sorted(train_pairs_all)

full_date_range = pd.date_range(train_fe["Date"].min(), test_fe["Date"].max(), freq="7D")
calendar_cols = [
    'IsHoliday', 'Year', 'Month', 'Week', 'DayOfYear', 'Quarter', 'Year_norm',
    'Week_sin', 'Week_cos', 'Month_sin', 'Month_cos',
    'weeks_to_SuperBowl', 'near_SuperBowl', 'before_SuperBowl',
    'weeks_to_LaborDay', 'near_LaborDay', 'before_LaborDay',
    'weeks_to_Thanksgiving', 'near_Thanksgiving', 'before_Thanksgiving',
    'weeks_to_Christmas', 'near_Christmas', 'before_Christmas',
    'is_super_bowl', 'is_labor_day', 'is_thanksgiving', 'is_christmas'
]
static_cols = ['Type_enc', 'Size_log']

store_covariates = pd.concat([
    train_fe[['Store', 'Date'] + calendar_cols + static_cols],
    test_fe[['Store', 'Date'] + calendar_cols + static_cols]
], axis=0).drop_duplicates(subset=['Store', 'Date']).sort_values(['Store', 'Date'])

target_rows = []
for store_i, dept_i in active_pairs:
    obs = train_fe[(train_fe.Store == store_i) & (train_fe.Dept == dept_i)][['Date', 'Weekly_Sales']]
    obs = obs.set_index('Date').reindex(full_date_range)
    obs['Store'] = store_i
    obs['Dept'] = dept_i
    obs['is_active'] = obs['Weekly_Sales'].notna().astype(int)
    obs['Weekly_Sales'] = obs['Weekly_Sales'].fillna(0).clip(lower=0)
    obs = obs.reset_index().rename(columns={'index': 'Date'})
    target_rows.append(obs)

target_panel = pd.concat(target_rows, axis=0, ignore_index=True)
full_panel = target_panel.merge(store_covariates, on=['Store', 'Date'], how='left')
full_panel = full_panel.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
full_panel['time_idx'] = full_panel.groupby(['Store', 'Dept']).cumcount()
full_panel['series_id'] = full_panel['Store'].astype(str) + "_" + full_panel['Dept'].astype(str)
full_panel['Store'] = full_panel['Store'].astype(str)
full_panel['Dept'] = full_panel['Dept'].astype(str)

MAX_ENCODER_LENGTH = 52
known_reals = calendar_cols
unknown_reals = ['Weekly_Sales']

def build_fold_datasets(fold_idx, batch_size=128):
    val_start, val_end = fold_cutoffs[fold_idx]
    training_data = full_panel[full_panel.time_idx < val_start]
    fold_data = full_panel[full_panel.time_idx < val_end]

    training_dataset = TimeSeriesDataSet(
        training_data, time_idx="time_idx", target="Weekly_Sales", group_ids=["series_id"],
        max_encoder_length=MAX_ENCODER_LENGTH, max_prediction_length=CV_FOLD_HORIZON,
        static_categoricals=["Store", "Dept"], static_reals=["Size_log"],
        time_varying_known_categoricals=[], time_varying_known_reals=known_reals,
        time_varying_unknown_categoricals=[], time_varying_unknown_reals=unknown_reals,
        target_normalizer=GroupNormalizer(groups=["series_id"], transformation="softplus"),
        weight="is_active", add_relative_time_idx=True, add_target_scales=True,
        add_encoder_length=True, allow_missing_timesteps=True,
    )
    validation_dataset = TimeSeriesDataSet.from_dataset(training_dataset, fold_data, predict=True, stop_randomization=True)
    train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)
    return training_dataset, validation_dataset, train_dataloader, val_dataloader

def compute_wmae(model, val_dataloader, full_panel, active_only=True):
    preds_out = model.predict(val_dataloader, mode="prediction", return_x=True, return_index=True)
    preds = preds_out.output.cpu().numpy()
    actuals = preds_out.x["decoder_target"].cpu().numpy()
    index_df = preds_out.index.reset_index(drop=True)
    horizon = preds.shape[1]
    rows = []
    for i in range(len(index_df)):
        series_id = index_df.loc[i, "series_id"]
        start_t = index_df.loc[i, "time_idx"]
        for h in range(horizon):
            rows.append({"series_id": series_id, "time_idx": start_t + h, "pred": preds[i, h], "actual": actuals[i, h]})
    pred_df = pd.DataFrame(rows)
    pred_df = pred_df.merge(full_panel[["series_id", "time_idx", "IsHoliday", "is_active"]], on=["series_id", "time_idx"], how="left")
    if active_only:
        pred_df = pred_df[pred_df["is_active"] == 1]
    weights = np.where(pred_df["IsHoliday"] == 1, 5, 1)
    wmae = np.average(np.abs(pred_df["pred"] - pred_df["actual"]), weights=weights)
    mae = np.mean(np.abs(pred_df["pred"] - pred_df["actual"]))
    return wmae, mae, pred_df

print("Panel and helper functions rebuilt. full_panel shape:", full_panel.shape)

In [ ]:
best_tft = TemporalFusionTransformer.load_from_checkpoint(local_checkpoint_path)
best_tft.eval()

_, _, _, val_dataloader_fold2 = build_fold_datasets(2)

wmae_check, mae_check, pred_df = compute_wmae(best_tft, val_dataloader_fold2, full_panel)
print(f"Confirmed WMAE from reload: {wmae_check:.2f} (should match logged 1772.43)")

series_avg = full_panel.groupby('series_id')['Weekly_Sales'].mean()
large_series = series_avg.idxmax()
small_candidates = series_avg[series_avg > 100].sort_values()
small_series = small_candidates.index[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sid, label in [(axes[0], large_series, "High-volume"), (axes[1], small_series, "Low-volume")]:
    subset = pred_df[pred_df['series_id'] == sid].sort_values('time_idx')
    if len(subset) == 0:
        ax.set_title(f"{label} — {sid} not in val set")
        continue
    ax.plot(subset['actual'].values, label="Actual", marker='o')
    ax.plot(subset['pred'].values, label="Predicted", marker='x')
    ax.set_title(f"{label} series — {sid}")
    ax.set_xlabel("Week into horizon")
    ax.legend()
plt.tight_layout()
plt.show()

residuals = pred_df['pred'] - pred_df['actual']
plt.figure(figsize=(9, 4))
sns.histplot(residuals, bins=80)
plt.axvline(0, color='red', linestyle='--')
plt.title("TFT Residual Distribution (Fold 2, Best CV Fold)")
plt.xlabel("Residual ($)")
plt.show()

holiday_mae = np.abs(pred_df[pred_df['IsHoliday']==1]['pred'] - pred_df[pred_df['IsHoliday']==1]['actual']).mean()
nonholiday_mae = np.abs(pred_df[pred_df['IsHoliday']==0]['pred'] - pred_df[pred_df['IsHoliday']==0]['actual']).mean()
print(f"\nHoliday MAE: {holiday_mae:.2f} | Non-holiday MAE: {nonholiday_mae:.2f}")

## Pipeline

In [ ]:
cold_start_pairs = list(set(zip(test_fe.Store, test_fe.Dept)) - set(zip(train_fe.Store, train_fe.Dept)))
dept_avg_fallback = train_fe.groupby('Dept')['Weekly_Sales'].mean()

print("Cold-start pairs:", len(cold_start_pairs))

class TFTRecursivePipeline(mlflow.pyfunc.PythonModel if 'mlflow' in dir() else object):
    def __init__(self, checkpoint_path, base_panel, active_pairs, cold_start_pairs, dept_avg_fallback,
                 calendar_cols, static_cols, max_encoder_length=52, step_horizon=13, n_steps=3):
        self.checkpoint_path = checkpoint_path
        self.base_panel = base_panel
        self.active_pairs = set(active_pairs)
        self.cold_start_pairs = set(cold_start_pairs)
        self.dept_avg_fallback = dept_avg_fallback
        self.calendar_cols = calendar_cols
        self.static_cols = static_cols
        self.max_encoder_length = max_encoder_length
        self.step_horizon = step_horizon
        self.n_steps = n_steps
        self._model = None

    def _load_model(self):
        if self._model is None:
            self._model = TemporalFusionTransformer.load_from_checkpoint(self.checkpoint_path)
            self._model.eval()
        return self._model

    def predict(self, context, model_input):
        test_df = model_input.copy()
        test_df['Date'] = pd.to_datetime(test_df['Date'])
        model = self._load_model()

        static_lookup = self.base_panel.drop_duplicates('series_id')[['series_id', 'Store', 'Dept'] + self.static_cols]
        working_panel = self.base_panel.copy()
        all_step_preds = []

        test_dates_sorted = sorted(test_df['Date'].unique())
        step_boundaries = [test_dates_sorted[i:i + self.step_horizon] for i in range(0, len(test_dates_sorted), self.step_horizon)]

        for step_dates in step_boundaries[:self.n_steps]:
            step_slice = test_df[test_df['Date'].isin(step_dates)].copy()

            series_max_idx = working_panel.groupby('series_id')['time_idx'].max().to_dict()

            future_rows = []
            for (store_i, dept_i), grp in step_slice.groupby(['Store', 'Dept']):
                sid = f"{store_i}_{dept_i}"
                if sid not in series_max_idx:
                    continue
                grp = grp.sort_values('Date').reset_index(drop=True)
                base_idx = series_max_idx[sid]
                for h, (_, row) in enumerate(grp.iterrows()):
                    future_rows.append({
                        'series_id': sid, 'Store': str(store_i), 'Dept': str(dept_i),
                        'Date': row['Date'], 'time_idx': base_idx + 1 + h,
                        'Weekly_Sales': 0.0, 'is_active': 1,
                        **{c: row[c] for c in self.calendar_cols}
                    })
            future_df = pd.DataFrame(future_rows)
            if len(future_df) == 0:
                break
            future_df = future_df.merge(static_lookup[['series_id'] + self.static_cols], on='series_id', how='left')

            step_panel = pd.concat([working_panel, future_df], ignore_index=True)
            step_panel = step_panel.drop_duplicates(subset=['series_id', 'time_idx'], keep='last')

            max_train_idx = working_panel['time_idx'].max()

            train_ds_step = TimeSeriesDataSet(
                step_panel[step_panel.time_idx <= max_train_idx],
                time_idx="time_idx", target="Weekly_Sales", group_ids=["series_id"],
                max_encoder_length=self.max_encoder_length, max_prediction_length=self.step_horizon,
                static_categoricals=["Store", "Dept"], static_reals=["Size_log"],
                time_varying_known_reals=self.calendar_cols, time_varying_unknown_reals=["Weekly_Sales"],
                target_normalizer=GroupNormalizer(groups=["series_id"], transformation="softplus"),
                weight="is_active", add_relative_time_idx=True, add_target_scales=True,
                add_encoder_length=True, allow_missing_timesteps=True,
            )
            predict_ds = TimeSeriesDataSet.from_dataset(train_ds_step, step_panel, predict=True, stop_randomization=True)
            predict_loader = predict_ds.to_dataloader(train=False, batch_size=256, num_workers=0)

            preds_out = model.predict(predict_loader, mode="prediction", return_x=True, return_index=True)
            preds = preds_out.output.cpu().numpy()
            index_df = preds_out.index.reset_index(drop=True)

            step_pred_rows = []
            for i in range(len(index_df)):
                sid = index_df.loc[i, 'series_id']
                start_t = index_df.loc[i, 'time_idx']
                for h in range(preds.shape[1]):
                    step_pred_rows.append({'series_id': sid, 'time_idx': start_t + h, 'Weekly_Sales_Pred': max(float(preds[i, h]), 0.0)})
            step_pred_df = pd.DataFrame(step_pred_rows)
            step_pred_df = step_pred_df.drop_duplicates(subset=['series_id', 'time_idx'], keep='first')
            all_step_preds.append(step_pred_df)

            pseudo_hist = future_df.merge(step_pred_df, on=['series_id', 'time_idx'], how='inner')
            pseudo_hist['Weekly_Sales'] = pseudo_hist['Weekly_Sales_Pred']
            pseudo_hist = pseudo_hist.drop(columns=['Weekly_Sales_Pred'])
            working_panel = pd.concat([working_panel, pseudo_hist], ignore_index=True)
            working_panel = working_panel.drop_duplicates(subset=['series_id', 'time_idx'], keep='last')

        final_preds = pd.concat(all_step_preds, ignore_index=True)
        final_preds = final_preds.drop_duplicates(subset=['series_id', 'time_idx'], keep='first')

        date_lookup = working_panel[['series_id', 'time_idx', 'Date', 'Store', 'Dept']].drop_duplicates(subset=['series_id', 'time_idx'])
        final_preds = final_preds.merge(date_lookup, on=['series_id', 'time_idx'], how='left')
        final_preds['Store'] = final_preds['Store'].astype(int)
        final_preds['Dept'] = final_preds['Dept'].astype(int)

        for store_i, dept_i in self.cold_start_pairs:
            fallback_val = self.dept_avg_fallback.get(dept_i, self.dept_avg_fallback.mean())
            rows = test_df[(test_df.Store == store_i) & (test_df.Dept == dept_i)]
            for _, row in rows.iterrows():
                final_preds = pd.concat([final_preds, pd.DataFrame([{
                    'Store': int(store_i), 'Dept': int(dept_i), 'Date': row['Date'], 'Weekly_Sales_Pred': fallback_val
                }])], ignore_index=True)

        return final_preds[['Store', 'Dept', 'Date', 'Weekly_Sales_Pred']]


print("Pipeline class defined (corrected: int dtypes, per-series time_idx, dedup at each merge step).")

In [ ]:
tft_pipeline = TFTRecursivePipeline(
    checkpoint_path=local_checkpoint_path,
    base_panel=full_panel,
    active_pairs=active_pairs,
    cold_start_pairs=cold_start_pairs,
    dept_avg_fallback=dept_avg_fallback,
    calendar_cols=calendar_cols,
    static_cols=static_cols,
    max_encoder_length=MAX_ENCODER_LENGTH,
    step_horizon=CV_FOLD_HORIZON,
    n_steps=3,
)

start_time = time.time()
tft_test_predictions = tft_pipeline.predict(None, test_fe)
elapsed = time.time() - start_time

print(f"Pipeline ran in {elapsed/60:.1f} minutes")
print("Prediction shape:", tft_test_predictions.shape)
print("Expected rows:", len(test_fe))
print(tft_test_predictions.head())

In [ ]:
tft_test_predictions_clean = (
    tft_test_predictions
    .groupby(['Store', 'Dept', 'Date'], as_index=False)['Weekly_Sales_Pred']
    .mean()
)

print("Cleaned shape:", tft_test_predictions_clean.shape)
print("Expected:", len(test_fe))

final_check = test_fe[['Store', 'Dept', 'Date']].merge(
    tft_test_predictions_clean, on=['Store', 'Dept', 'Date'], how='left'
)
print("\nAny missing predictions after cleanup:", final_check['Weekly_Sales_Pred'].isnull().sum())
print("Any negative predictions:", (final_check['Weekly_Sales_Pred'] < 0).sum())

In [ ]:
with mlflow.start_run(run_name="TFT_Final_Pipeline"):
    mlflow.set_tags({"model_type": "TFT", "run_type": "final_pipeline", "method": "recursive_3step"})
    mlflow.log_params({
        "source_fold": 2, "source_run_id": "6d311265d3aa495f934d60485628816d",
        "step_horizon": CV_FOLD_HORIZON, "n_steps": 3, "max_encoder_length": MAX_ENCODER_LENGTH,
        "cv_val_wmae_fold2": 1772.43, "cold_start_fallback": "dept_average",
    })
    mlflow.log_metric("output_rows", len(final_check))
    mlflow.log_metric("negative_predictions", int((final_check['Weekly_Sales_Pred'] < 0).sum()))

    final_check.to_csv('tft_test_predictions.csv', index=False)
    mlflow.log_artifact('tft_test_predictions.csv')

    example_input = test_fe[['Store', 'Dept', 'Date'] + calendar_cols].head(5)
    mlflow.pyfunc.log_model(
        python_model=tft_pipeline,
        name="model",
        input_example=example_input,
    )

print("\nTFT pipeline logged to MLflow (not registered — TFT is the weakest performer, registration reserved for the final cross-architecture winner)")